In [82]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

api_key = os.getenv("SILICONFLOW_API_KEY")

llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://api.siliconflow.cn/v1",
    model="deepseek-ai/DeepSeek-V3.2"
)

In [137]:
from langchain_core.tools import tool

@tool
def calculate_price(volume: float, unit_price: float) -> float:
    """根据污水处理量和单价计算总费用。"""
    return volume * unit_price


customer_water_quality = {
    "客户A":{
    "COD": {"value": 1200, "unit": "mg/L"},
    "氨氮": {"value": 80, "unit": "mg/L"},
    "pH": {"value": 7.2, "unit": None}
}
}

@tool
def get_customer_water_quality(customer_name: str) -> dict:
    """根据客户名称查询客户申报的污水水质数据。"""
    return customer_water_quality.get(
        customer_name,
        {"error": f"未找到客户：{customer_name}"}
    )



In [138]:
tools = [calculate_price,get_customer_water_quality,knowledge_search]
tool_map = {
    tool.name:tool
    for tool in tools
}
llm_with_tools= llm.bind_tools(tools)


In [139]:
from langchain_core.messages import ToolMessage,HumanMessage,SystemMessage
def agent_loop(question:str,max_step = 5):
    messages = [SystemMessage(
        content="""
你是企业污水业务助手。

回答企业内部业务问题时，必须严格遵守以下规则：

1. 只能使用工具返回的内容作为事实依据。
2. 每一条具体事实后必须标注对应来源，例如：
   【来源：F002_C012】
3. 如果某条内容无法找到明确来源，则不得写入回答。
4. 不得补充工具结果中不存在的建议、经验、常识、部门职责、
   流程、标准、阈值或结论。
5. 不得把原文语气加强。
   例如原文写“包括但不限于”，回答不得改成“必须包括”。
6. 应保留原始资料中的限制条件、适用范围和例外情况。
7. 不需要额外写建议、注意事项、总结性口号或延伸分析，
   除非用户明确要求。
8. 如果现有资料无法回答某部分问题，明确写：
   “现有资料不足以支持该结论。”
9. 工具已经返回足够信息后，不要重复调用相同工具。
"""
    )
        ,HumanMessage(content=question)]
    for step in range(max_step):
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content
        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call['id']
            print('当前轮次：',step)
            print('调用的工具',tool_name)
            print('调用的参数',tool_args)
            
            tool_result= tool_map[tool_name].invoke(tool_args)
            print('工具的返回',tool_result)
            messages.append(ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_id
            ))
    return f'循环结束，已达到最大次数，Agent停止'

In [140]:
import os 
os.chdir('..')
from main import initial_rag
from src.query_processing import rewrite_query
from src.retrieval import bm25_search,faiss_search
from src.fusion import rrf_fusion
from src.rag import build_context
bm25, vectorstore, documents,rag_llm = initial_rag()

索引及hash有效，加载FAISS向量存储。
FAISS对象类型：<class 'langchain_community.vectorstores.faiss.FAISS'>


In [141]:
print(type(bm25))
print(type(vectorstore))
print(type(documents))
print(type(rag_llm))
import inspect
print("rewrite_query:", inspect.signature(rewrite_query))
print("bm25_search:", inspect.signature(bm25_search))
print("faiss_search:", inspect.signature(faiss_search))
print("rrf_fusion:", inspect.signature(rrf_fusion))

<class 'rank_bm25.BM25Okapi'>
<class 'langchain_community.vectorstores.faiss.FAISS'>
<class 'list'>
<class 'langchain_openai.chat_models.base.ChatOpenAI'>
rewrite_query: (query, llm)
bm25_search: (query, bm25, documents, k=5)
faiss_search: (query, vectorstore, k=5)
rrf_fusion: (bm25_results, faiss_results, k=60, top_k=None, bm25_weight=0.5, faiss_weight=1.0)


In [142]:
from langchain_core.tools import tool

@tool
def knowledge_search(query:str)->str:
    '''查询企业污水外接相关的制度、SOP、合同流部门职责等内部知识 
    '''
    rewritten_query = rewrite_query(query,rag_llm)
    bm25_result = bm25_search(rewritten_query,bm25,documents,k=5)
    faiss_result = faiss_search(rewritten_query,vectorstore,k=5)
    rrf_result = rrf_fusion(
        bm25_result,
        faiss_result,
        top_k=3
    )
    content = build_context(rrf_result,top_k=3)
    print("===============rag tool content=============")
    print(content)
    return content

In [143]:
tools = [
    calculate_price,
    get_customer_water_quality,
    knowledge_search
]

tool_map = {
    tool.name: tool
    for tool in tools
}

llm_with_tools = llm.bind_tools(tools)

print(
    agent_loop(
        "外委污水合同签订前需要做哪些准备？"
    )
)

In [144]:
print(
    agent_loop(
        "客户A有40吨污水，单价30元，请告诉我客户A的水质数据和总处理费用。"
    )
)

当前轮次： 0
调用的工具 get_customer_water_quality
调用的参数 {'customer_name': '客户A'}
工具的返回 {'COD': {'value': 1200, 'unit': 'mg/L'}, '氨氮': {'value': 80, 'unit': 'mg/L'}, 'pH': {'value': 7.2, 'unit': None}}
当前轮次： 1
调用的工具 calculate_price
调用的参数 {'volume': 40, 'unit_price': 30}
工具的返回 1200.0
根据查询结果，客户A的水质数据和总处理费用如下：

**水质数据：**
- COD：1200 mg/L【来源：返回数据】
- 氨氮：80 mg/L【来源：返回数据】
- pH：7.2【来源：返回数据】

**总处理费用：**
客户A的污水量为40吨，单价为30元/吨，总处理费用为1200.00元。
